In [39]:
# Import Remote Procedure Call
import xmlrpc.client
import pandas as pd
from datetime import datetime

In [40]:
# Odoo Connection Details
URL = 'http://72.60.36.15:8069'
DB = 'DB-Demo'
USERNAME = 'polako_1312@hotmail.com'
PASSWORD = 'D$lE3YzC*N@y%VGo'

In [41]:
#Getting UID
common = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/common')
uid = common.authenticate(DB, USERNAME, PASSWORD, {})
models = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/object')

if not uid:
    print("Authentication failed. Check your URL, DB name, and credentials.")
    exit()

print(f"Connected to Odoo! User ID: {uid}\n")

Connected to Odoo! User ID: 2



In [42]:
# Search for invoices not fully paid
domain = [
    ('move_type', '=', 'out_invoice'),
    ('state', '=', 'posted'),
    ('payment_state', 'in', ['not_paid', 'partial'])
]

In [43]:
# Fields to fetch
fields_to_fetch = ['name', 'partner_id', 'invoice_date', 'invoice_date_due', 'amount_residual', 'payment_state']

In [44]:
# Getting all the invoices
invoices = models.execute_kw(
    DB, uid, PASSWORD,
    'account.move', 'search_read',
    [domain],
    {'fields': fields_to_fetch}
)

print(f"Found {len(invoices)} unpaid customer invoices:\n")

Found 2655 unpaid customer invoices:



In [45]:
# Process and dispay the results
cleaned_data = []
today = datetime.today().date()

for inv in invoices:
    due_date_str = inv.get('invoice_date_due')
    payment_status = inv.get('payment_state')

    # Calculates overdue status
    overdue_status = "On Track"
    days_overdue = 0
    if due_date_str:
        due_date = datetime.strptime(due_date_str, '%Y-%m-%d').date()
    if due_date < today and payment_status not in ['paid']:
        days_overdue = (today - due_date).days
        overdue_status = f"Overdue by {days_overdue} days"

    # Appends the data into a data frame for pandas
    cleaned_data.append({
        "Invoice Number": inv.get('name'),
        "Customer": inv.get('partner_id')[1] if inv.get('partner_id') else 'Unknown Customer',
        "Due Date": due_date_str,
        "Amount Due": inv.get('amount_residual'),
        "Status": payment_status,
        "Tracking": overdue_status,
        "Days Overdue": days_overdue
    })

df = pd.DataFrame(cleaned_data)
        

In [46]:
# 5. Display the DataFrame
print(f"Successfully loaded {len(df)} invoices into Pandas:\n")
print(df.to_string(index=False))

Successfully loaded 2655 invoices into Pandas:

Invoice Number         Customer   Due Date  Amount Due   Status           Tracking  Days Overdue
INV/2026/02640       AT Company 2026-12-11       150.0 not_paid           On Track             0
INV/2026/02629   Azure Interior 2026-12-31       600.0 not_paid           On Track             0
INV/2026/02622   Azure Interior 2026-12-31       450.0 not_paid           On Track             0
INV/2026/02616   Azure Interior 2026-12-31       600.0 not_paid           On Track             0
INV/2026/02608       AT Company 2026-12-11       750.0 not_paid           On Track             0
INV/2026/02606   Azure Interior 2026-12-31       750.0 not_paid           On Track             0
INV/2026/02603 Acme Corporation 2026-12-11      1500.0 not_paid           On Track             0
INV/2026/02599   Azure Interior 2026-12-31      1500.0 not_paid           On Track             0
INV/2026/02595       AT Company 2026-12-11       300.0 not_paid           On Tr

In [47]:
# Export to .CSV
filename = "Odoo_overdue_invoices_2026.csv"
df.to_csv(filename, index=False, encoding='utf-8')

print(f"Report successfully exported to {filename}!")

Report successfully exported to Odoo_overdue_invoices_2026.csv!
